# FrameSat AI: Experiment 001
**Objective:** Fine-tuning Practical-RIFE 4.26 on GOES-19
**Project Name:** FrameSat-AI
**Experiment Number:** 001
**Model:** Practical-RIFE 4.26
**Dataset:** GOES-19 ABI-L1b-RadC (CONUS)
**Hardware:** NVIDIA GPU
**Expected Outputs:** Fine-tuned weights, evaluation metrics, logs, and visual predictions.


In [1]:
import os
import subprocess
import sys

# Dynamically discover bundle path for installing requirements
bundle_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "training" in dirs and "evaluation" in dirs:
        bundle_path = root
        break

if not bundle_path:
    if os.path.exists("/kaggle/working/training/train.py"):
        bundle_path = "/kaggle/working"

if bundle_path:
    req_path = os.path.join(bundle_path, "kaggle", "requirements.txt")
    print(f"Installing requirements from: {req_path}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", req_path])
else:
    print("Warning: Bundle not found. Installing key packages directly.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "piq"])


Installing requirements from: /kaggle/input/datasets/nishanthpalugula/framesat-ai-training-bundle/kaggle/requirements.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 kB 3.0 MB/s eta 0:00:00


## 2. Environment Verification


In [2]:
import sys
import platform
import psutil
import torch
import os
import json
import glob
import subprocess
import sqlite3

print(f"Python version: {sys.version.split(' ')[0]}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("Error: CUDA not available. Aborting.")
    raise RuntimeError("Execution aborted")
    
print(f"CPU cores: {psutil.cpu_count()}")
print(f"RAM: {psutil.virtual_memory().total / 1024**3:.2f} GB")



Python version: 3.12.13
PyTorch version: 2.10.0+cu128
CUDA version: 12.8
GPU: Tesla T4
GPU Memory: 14.56 GB
CPU cores: 4
RAM: 31.35 GB


## 3. Dataset Discovery


In [1]:

import os
import sys
import glob

# 1. Discover the bundle path (look for training/train.py)
bundle_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "training" in dirs and "evaluation" in dirs:
        bundle_path = root
        break

if not bundle_path:
    # Try working directory as fallback
    if os.path.exists("/kaggle/working/training/train.py"):
        bundle_path = "/kaggle/working"

if not bundle_path:
    print("Error: Could not find framesat-ai-training-bundle in /kaggle/input or /kaggle/working.")
    print("Please ensure you have added the bundle dataset.")
    raise RuntimeError("Execution aborted")

# 2. Discover the dataset path (look for cache directory)
dataset_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "cache" in dirs and "quarantine" in dirs:
        dataset_path = root
        break

if not dataset_path:
    # Fallback to just cache
    for root, dirs, files in os.walk("/kaggle/input"):
        if "cache" in dirs:
            dataset_path = root
            break

if not dataset_path:
    print("Error: Could not find GOES-19 dataset containing a 'cache' folder in /kaggle/input.")
    print("Please ensure you have added the dataset to your Kaggle Notebook.")
    raise RuntimeError("Execution aborted")

if bundle_path not in sys.path:
    sys.path.insert(0, bundle_path)

cache_dir = os.path.join(dataset_path, "cache")
metadata_db = os.path.join(dataset_path, "metadata.db")
reports_dir = os.path.join(dataset_path, "reports")

nc_files = glob.glob(os.path.join(cache_dir, "*.nc"))
num_scenes = len(nc_files)
cache_size_gb = sum(os.path.getsize(f) for f in nc_files) / 1024**3 if nc_files else 0

reports_found = len(os.listdir(reports_dir)) if os.path.exists(reports_dir) else 0

print(f"Bundle Path: {bundle_path}")
print(f"Dataset Path: {dataset_path}")
print(f"Number of Scenes (Cache): {num_scenes}")
print(f"Cache Size: {cache_size_gb:.2f} GB")
print(f"Metadata DB Status: {'Found' if os.path.exists(metadata_db) else 'Missing'}")
print(f"Reports Found: {reports_found}")

Bundle Path: /kaggle/input/datasets/nishanthpalugula/framesat-ai-training-bundle
Dataset Path: /kaggle/input/datasets/nishanthpalugula/framesat-goes-19-v1/framesat-goes19-v1
Number of Scenes (Cache): 1015
Cache Size: 3.94 GB
Metadata DB Status: Found
Reports Found: 10


## 4. Dataset Summary


In [3]:
import json
import sqlite3
import os
import pandas as pd

stats_path = os.path.join(dataset_path, "dataset_statistics.json")
stats = {}
if os.path.exists(stats_path):
    with open(stats_path, 'r') as f:
        stats = json.load(f)

# Optional: read from metadata.db if needed
# try:
#     conn = sqlite3.connect(metadata_db)
#     cursor = conn.cursor()
#     # fetch whatever is needed
#     conn.close()
# except Exception:
#     pass

summary_data = {
    "Metric": ["Scene Count", "Triplet Count", "Channel", "Sector", "Temporal Coverage", "Resolution", "Dataset Size (GB)"],
    "Value": [
        num_scenes,
        max(0, num_scenes - 2),
        stats.get("channel", 13),
        stats.get("sector", "CONUS"),
        f"{stats.get('start_time', 'Unknown')} to {stats.get('end_time', 'Unknown')}",
        stats.get("spatial_resolution", "Unknown"),
        f"{cache_size_gb:.2f}"
    ]
}

df_summary = pd.DataFrame(summary_data)
display(df_summary.style.hide(axis='index'))



Metric,Value
Scene Count,1015
Triplet Count,1013
Channel,13
Sector,CONUS
Temporal Coverage,Unknown to Unknown
Resolution,Unknown
Dataset Size (GB),3.94


## 5. Configuration


In [4]:
config_path = os.path.join(bundle_path, "kaggle", "train_kaggle.json")
with open(config_path, 'r') as f:
    config = json.load(f)

config["dataset_path"] = cache_dir
config["quarantine_dir"] = os.path.join(dataset_path, "quarantine")

# Robust path resolution for weights in Kaggle bundle
pretrained_rel = config.get("pretrained_weights", "weights/rife_426/train_log")
if not os.path.exists(os.path.join(bundle_path, pretrained_rel)):
    if os.path.exists(os.path.join(bundle_path, "evaluation", pretrained_rel)):
        pretrained_rel = os.path.join("evaluation", pretrained_rel)

config["pretrained_weights"] = os.path.join(bundle_path, pretrained_rel)

print("Resolved Configuration:")
print(json.dumps(config, indent=4))

# Write overridden config to working dir
os.makedirs("/kaggle/working/configs", exist_ok=True)
working_config_path = "/kaggle/working/configs/train_kaggle_resolved.json"
with open(working_config_path, 'w') as f:
    json.dump(config, f, indent=4)


Resolved Configuration:
{
    "experiment_name": "kaggle_rife426_finetune",
    "dataset_path": "/kaggle/input/datasets/nishanthpalugula/framesat-goes-19-v1/framesat-goes19-v1/cache",
    "quarantine_dir": "/kaggle/input/datasets/nishanthpalugula/framesat-goes-19-v1/framesat-goes19-v1/quarantine",
    "pretrained_weights": "/kaggle/input/datasets/nishanthpalugula/framesat-ai-training-bundle/evaluation/weights/rife_426/train_log",
    "epochs": 10,
    "batch_size": 4,
    "learning_rate": 0.0001,
    "loss_alpha": 0.5,
    "use_amp": true,
    "resume": false,
    "resume_checkpoint": "",
    "log_interval": 50,
    "save_visual_interval": 1,
    "lr_scheduler": "cosine",
    "early_stopping_patience": 5,
    "seed": 42,
    "train_resize": [
        384,
        384
    ],
    "start_date": "2024-10-10T21:00:00",
    "end_date": "2024-10-14T09:00:00"
}


## 6. Pre-Flight Checks


In [5]:

import shutil

checks_passed = True

# 1. Weights exist
weight_file = os.path.join(config["pretrained_weights"], "flownet.pkl")
if not os.path.exists(weight_file):
    print(f"[FAIL] Pretrained weights not found: {weight_file}")
    checks_passed = False
else:
    print(f"[PASS] Pretrained weights found.")

# 2. Dataset exists & Triplets > 0
if num_scenes < 3:
    print(f"[FAIL] Insufficient scenes ({num_scenes}) for triplets.")
    checks_passed = False
else:
    print(f"[PASS] Triplets available (>0).")

# 3. Disk space sufficient (Assume Kaggle has ~20GB in /kaggle/working)
total, used, free = shutil.disk_usage("/kaggle/working/")
free_gb = free / 1024**3
if free_gb < 5.0:
    print(f"[FAIL] Insufficient disk space in /kaggle/working: {free_gb:.2f} GB free")
    checks_passed = False
else:
    print(f"[PASS] Disk space sufficient: {free_gb:.2f} GB free")

# 4. Output directory writable
try:
    test_file = "/kaggle/working/test_write.tmp"
    with open(test_file, 'w') as f:
        f.write("test")
    os.remove(test_file)
    print("[PASS] Output directory is writable.")
except Exception as e:
    print(f"[FAIL] Output directory not writable: {e}")
    checks_passed = False

if not checks_passed:
    print("\nError: Pre-flight checks failed. Aborting.")
    raise RuntimeError("Execution aborted")



[PASS] Pretrained weights found.
[PASS] Triplets available (>0).
[PASS] Disk space sufficient: 19.50 GB free
[PASS] Output directory is writable.


## 7. Training


In [8]:

# Execute the training pipeline using the resolved config and dynamic bundle path
!python {bundle_path}/training/train.py --config /kaggle/working/configs/train_kaggle_resolved.json


Kaggle environment detected. Auto-mapping dataset paths...
Mapped dataset via search to: /kaggle/input/datasets/nishanthpalugula/framesat-goes-19-v1/framesat-goes19-v1

GOES-19 DATASET SUMMARY

Scenes:
1015

Valid scenes:
1015

Rejected:
0

Triplets:
1013

Channel:
13

Sector:
CONUS

Resolution:
1500 x 2500

Training Resolution:
384 x 384

Dataset Size:
3.94 GB

Traceback (most recent call last):
  File "/kaggle/input/datasets/nishanthpalugula/framesat-ai-training-bundle/training/train.py", line 191, in <module>
    main()
  File "/kaggle/input/datasets/nishanthpalugula/framesat-ai-training-bundle/training/train.py", line 157, in main
    train_dataset = GOES19TripletDataset(
                    ^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/input/datasets/nishanthpalugula/framesat-ai-training-bundle/datasets/providers/goes19/goes19_builder.py", line 120, in __init__
    with open(metadata_path, 'w') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 30] Read-only file system: '/kaggle/inp

## 8. Live Monitoring


To monitor training with TensorBoard:

```python
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/artifacts/training/runs
```

Since Kaggle clears notebook outputs occasionally, the training script outputs logs directly to stdout (Epoch, Train Loss, Validation Metrics, ETA).



## 9. Evaluation


In [ ]:
# Automatically evaluate the best checkpoint after training
# Find the run directory
import time

runs_dir = "/kaggle/working/artifacts/training/runs"
if not os.path.exists(runs_dir):
    print("No training runs found. Skipping evaluation.")
else:
    # Find latest run
    all_runs = sorted(glob.glob(os.path.join(runs_dir, "run_*")))
    if not all_runs:
         print("No runs found.")
    else:
        latest_run = all_runs[-1]
        best_checkpoint = os.path.join(latest_run, "best.pth")
        
        if not os.path.exists(best_checkpoint):
            # Fallback to checking the models/checkpoints folder
            best_checkpoint = os.path.join(latest_run, "flownet.pkl")
            
        if not os.path.exists(best_checkpoint):
            print(f"Best checkpoint not found at {best_checkpoint}. Skipping evaluation.")
        else:
            print(f"Evaluating best checkpoint: {best_checkpoint}")
            
            # Setup evaluation config
            eval_config = {
                "experiment_name": "eval_kaggle_finetune",
                "dataset_path": cache_dir,
                "weights": best_checkpoint,
                "events": 10,
                "save_predictions": False
            }
            
            eval_config_path = "/kaggle/working/configs/eval_config.json"
            with open(eval_config_path, 'w') as f:
                json.dump(eval_config, f, indent=4)
                
            # Extract start and end dates from config for dataset constructor
            start_date_str = config.get("start_date", "2024-10-10T21:00:00")
            end_date_str = config.get("end_date", "2024-10-14T09:00:00")
            
            # Run evaluator script using the correct RIFEInterpolator interface and GOES19EvaluationDatasetWrapper
            runner_code = f"""
import sys
import os
import json
import torch
from datetime import datetime
sys.path.insert(0, '{bundle_path}')

from evaluation.evaluator import Evaluator
from datasets.providers.goes19.goes19_builder import GOES19TripletDataset
from models.rife.interpolator import RIFEInterpolator

# 1. Wrapper to make GOES19TripletDataset compatible with Evaluator API
class GOES19EvaluationDatasetWrapper:
    def __init__(self, goes_dataset):
        self.dataset = goes_dataset
        self.modality = "vis"
        
    def load(self):
        pass
        
    @property
    def num_events(self):
        return len(self.dataset)
        
    def get_event_triplet(self, idx):
        t0, gt, t2 = self.dataset[idx]
        return t0.squeeze(0).numpy(), gt.squeeze(0).numpy(), t2.squeeze(0).numpy()

# 2. Initialize RIFEInterpolator wrapper
model = RIFEInterpolator()
# Pass the directory containing flownet.pkl
model.load_weights('{latest_run}')

# 3. Instantiate GOES19TripletDataset
raw_dataset = GOES19TripletDataset(
    start_date=datetime.fromisoformat('{start_date_str}'),
    end_date=datetime.fromisoformat('{end_date_str}'),
    cache_dir='{cache_dir}',
    split='val'
)
dataset = GOES19EvaluationDatasetWrapper(raw_dataset)

# 4. Run Evaluator
with open('{eval_config_path}', 'r') as f:
    eval_config = json.load(f)
    
evaluator = Evaluator(model, dataset, eval_config)
evaluator.run()
"""
            runner_script_path = "/kaggle/working/run_eval.py"
            with open(runner_script_path, 'w') as f:
                f.write(runner_code)
                
            !python {runner_script_path}


## 10. Baseline Comparison


In [ ]:

baseline_csv = os.path.join(bundle_path, "evaluation", "experiments", "experiments_summary.csv")
finetuned_csv = "/kaggle/working/evaluation/experiments/experiments_summary.csv"

if os.path.exists(baseline_csv) and os.path.exists(finetuned_csv):
    try:
        df_base = pd.read_csv(baseline_csv)
        df_base = df_base[df_base['Experiment'].str.contains('baseline', case=False, na=False)].tail(1)
        
        df_fine = pd.read_csv(finetuned_csv).tail(1)
        
        if not df_base.empty and not df_fine.empty:
            b_psnr = float(df_base['Avg_PSNR'].values[0])
            b_ssim = float(df_base['Avg_SSIM'].values[0])
            b_fsim = float(df_base['Avg_FSIM'].values[0])
            b_mse = float(df_base['Avg_MSE'].values[0])
            
            f_psnr = float(df_fine['Avg_PSNR'].values[0])
            f_ssim = float(df_fine['Avg_SSIM'].values[0])
            f_fsim = float(df_fine['Avg_FSIM'].values[0])
            f_mse = float(df_fine['Avg_MSE'].values[0])
            
            comp_data = {
                "Metric": ["PSNR", "SSIM", "FSIM", "MSE"],
                "Baseline": [b_psnr, b_ssim, b_fsim, b_mse],
                "Fine-tuned": [f_psnr, f_ssim, f_fsim, f_mse],
                "Δ": [f_psnr - b_psnr, f_ssim - b_ssim, f_fsim - b_fsim, f_mse - b_mse]
            }
            
            df_comp = pd.DataFrame(comp_data)
            def highlight_improvement(row):
                styles = [''] * len(row)
                delta = row['Δ']
                metric = row['Metric']
                is_better = delta > 0 if metric in ['PSNR', 'SSIM', 'FSIM'] else delta < 0
                if is_better:
                    styles[-1] = 'color: green; font-weight: bold'
                else:
                    styles[-1] = 'color: red'
                return styles
                
            display(df_comp.style.apply(highlight_improvement, axis=1).hide(axis='index'))
        else:
            print("Baseline or Fine-tuned metrics row not found in CSVs.")
    except Exception as e:
        print(f"Could not generate baseline comparison: {e}")
else:
    print("Baseline or Fine-tuned CSV not found. Cannot perform comparison.")



## 11. Visual Results


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Find the latest visualization directory
eval_runs_dir = "/kaggle/working/evaluation/experiments"
if os.path.exists(eval_runs_dir):
    all_evals = sorted(glob.glob(os.path.join(eval_runs_dir, "run_*")))
    if all_evals:
        latest_eval = all_evals[-1]
        vis_dir = os.path.join(latest_eval, "visualizations")
        
        if os.path.exists(vis_dir):
            vis_images = sorted(glob.glob(os.path.join(vis_dir, "*_vis.png")))
            for vis_img in vis_images[:5]: # Show first 5
                img = mpimg.imread(vis_img)
                plt.figure(figsize=(20, 10))
                plt.imshow(img)
                plt.axis('off')
                # Title should already be embedded in the *_vis.png by evaluator
                plt.show()
        else:
            print("Visualization directory not found.")
    else:
        print("No evaluation runs found for visualization.")
else:
    print("Evaluation directory not found.")



## 12. Export


In [ ]:

import tarfile

# Compress outputs
output_tar = "/kaggle/working/experiment_001_outputs.tar.gz"
print(f"Compressing outputs to {output_tar}...")

with tarfile.open(output_tar, "w:gz") as tar:
    if os.path.exists("/kaggle/working/artifacts"):
        tar.add("/kaggle/working/artifacts", arcname="artifacts")
    if os.path.exists("/kaggle/working/evaluation"):
        tar.add("/kaggle/working/evaluation", arcname="evaluation")
    if os.path.exists("/kaggle/working/configs"):
        tar.add("/kaggle/working/configs", arcname="configs")

print("Export complete. You can download the tar.gz file from the Kaggle output pane.")



## 13. Final Summary


In [ ]:

from IPython.display import display, Markdown

summary_md = f"""
### Experiment 001 Report

- **Dataset Version**: GOES-19 v1
- **Model Version**: Practical-RIFE 4.26
- **Checkpoints**: `/kaggle/working/artifacts/training/runs/`
- **Metrics**: `/kaggle/working/evaluation/experiments/`

**Recommendations for Experiment 002**:
- Analyze difference heatmaps to identify specific edge cases where interpolation fails.
- Consider tuning the loss_alpha parameter if artifacts appear in high-motion regions.
- Evaluate with a longer temporal stride to test robustness.
"""
display(Markdown(summary_md))

